# Deep dive into Transformers, Quantization and Neural Networks: [Google Colab Notebook Copy]

## Models:

Today we are working on lower level API of Transformers: Models that wrap PyTorch code for the transformers themselves.

* "bitsandbyte": Huggingface library which helps with Quantization.
* "accelerate": Helps to run task more efficiently on GPUs.

In [ ]:
# !pip install -q --upgrade bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.9 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch
import gc

In [3]:
hf_key = userdata.get("HuggingFace")
login(hf_key, add_to_git_credential = True)

In [4]:
# instruct models and 1 reasoning model

# Llama 3.1 is larger and you should already be approved
# see here: https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct

# LLAMA = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# Llama 3.2 is smaller but you might need to request access again
# see here: https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct

LLAMA = "meta-llama/Llama-3.2-1B-Instruct"

PHI = "microsoft/Phi-4-mini-instruct"
GEMMA = "google/gemma-3-270m-it"
QWEN = "Qwen/Qwen3-4B-Instruct-2507"
DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

In [5]:
messages = [
    {"role":"user","content":"Tell me a joke about a Machine Learning Engineer!"}
]

## Setting up Quantization Configuration:

In [30]:
quant_config = BitsAndBytesConfig(
    load_int_4bit = True,
    bnb_4bit_use_double_quant = True, # perform quantization twice
    bnb_4bit_compute_dtype = torch.bfloat16,
    bnb_4bit_quant_type = "nf4" # use NF4 weight precision format
)

## Setting up Tokenizer:

In [7]:
tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda") # return_tensor = "pt" means that the inputs should be returned in pytorch tensor format!

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

## Calling a HuggingFace Model using "AutoModelCausalLM" method

Just like AutoTokenizer to call tokenizer models, AutoModelCausalLM function is used to call Models from huggingface.

In [8]:
model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map = "auto", quantization_config = quant_cofig)

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [9]:
memory = model.get_memory_footprint() / 1e6
print(f" Memory: {memory:,.1f} MB")

 Memory: 1,012.0 MB


## Looking under the hood at the Transformer model

The next cell prints the HuggingFace `model` object for Llama.

This model object is a Neural Network, implemented with the Python framework PyTorch. The Neural Network uses the architecture invented by Google scientists in 2017: the Transformer architecture.

While we're not going to go deep into the theory, this is an opportunity to get some intuition for what the Transformer actually is.

If you're completely new to Neural Networks, check out my [YouTube intro playlist](https://www.youtube.com/playlist?list=PLWHe-9GP9SMMdl6SLaovUQF2abiLGbMjs) for the foundations.

Now take a look at the layers of the Neural Network that get printed in the next cell. Look out for this:

- It consists of layers
- There's something called "embedding" - this takes tokens and turns them into 4,096 dimensional vectors. We'll learn more about this in Week 5.
- There are then 16 sets of groups of layers (32 for Llama 3.1) called "Decoder layers". Each Decoder layer contains three types of layer: (a) self-attention layers (b) multi-layer perceptron (MLP) layers (c) batch norm layers.
- There is an LM Head layer at the end; this produces the output

Notice the mention that the model has been quantized to 4 bits.

It's not required to go any deeper into the theory at this point, but if you'd like to, I've asked our mutual friend to take this printout and make a tutorial to walk through each layer. This also looks at the dimensions at each point. If you're interested, work through this tutorial after running the next cell:

https://chatgpt.com/canvas/shared/680cbea6de688191a20f350a2293c76b

In [10]:
model # displays the structure/architecture of the model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm

## Running the model:

In [15]:
# Fix: Unpack the tokenizer output using **
outputs = model.generate(**inputs, max_new_tokens=80)

# Decode the generated tokens back to text
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_text)

system

Cutting Knowledge Date: December 2023
Today Date: 23 Oct 2026

user

Tell me a joke about a Machine Learning Engineer!assistant

A Machine Learning Engineer walked into a bar and ordered a beer. As he was sipping his drink, he heard a voice say, "You're a long way from home." He looked around, but he couldn't see anyone. A few minutes later, he heard, "No, you're just a model for me."


In [19]:
outputs[0]

tensor([128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
            25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
           220,   1419,   5020,    220,   2366,     21,    271, 128009, 128006,
           882, 128007,    271,  41551,    757,    264,  22380,    922,    264,
         13257,  21579,  29483,      0, 128009, 128006,  78191, 128007,    271,
            32,  13257,  21579,  29483,  15203,   1139,    264,   3703,    323,
         11713,    264,  13179,     13,   1666,    568,    574,    274,   5772,
           813,   7172,     11,    568,   6755,    264,   7899,   2019,     11,
           330,   2675,   2351,    264,   1317,   1648,    505,   2162,   1210,
          1283,   7111,   2212,     11,    719,    568,   7846,    956,   1518,
          5606,     13,    362,   2478,   4520,   3010,     11,    568,   6755,
            11,    330,   2822,     11,    499,   2351,   1120,    264,   1646,
           369,    757,   1210, 128009],

# Wrapping everything in a function:

In [26]:
# Wrapping everything in a function - and adding Streaming and generation prompts

# def generate(model, messages, quant=True, max_new_tokens=80):
#   tokenizer = AutoTokenizer.from_pretrained(model)
#   tokenizer.pad_token = tokenizer.eos_token
#   input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")
#   attention_mask = torch.ones_like(**input_ids, dtype=torch.long, device="cuda")
#   streamer = TextStreamer(tokenizer)
#   if quant:
#     model = AutoModelForCausalLM.from_pretrained(model, quantization_config=quant_config).to("cuda")
#   else:
#     model = AutoModelForCausalLM.from_pretrained(model).to("cuda")
#   outputs = model.generate(**input_ids, attention_mask=attention_mask, max_new_tokens=max_new_tokens, streamer=streamer)



In [31]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer

def generate(model_name_or_path, messages, quant=True, max_new_tokens=80):
    tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # 1. return_dict=True returns a dictionary with input_ids AND attention_mask automatically
    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        return_dict=True,
        add_generation_prompt=True
    ).to("cuda")

    streamer = TextStreamer(tokenizer)

    # 2. Use device_map="auto" instead of .to("cuda") for quantized loading
    if quant:
        model = AutoModelForCausalLM.from_pretrained(
            model_name_or_path,
            quantization_config=quant_config,
            device_map="auto"
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_name_or_path,
            device_map="auto"
        )

    # 3. Unpack input_ids and attention_mask directly into model.generate()
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        streamer=streamer,
        pad_token_id=tokenizer.pad_token_id
    )
    return outputs

In [32]:
 generate(PHI, messages)

model.safetensors.index.json:   0%|          | 0.00/16.3k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

<|user|>Tell me a joke about a Machine Learning Engineer!<|end|><|assistant|>Sure, here's a light-hearted joke about a Machine Learning Engineer:

Why did the Machine Learning Engineer break up with the computer?

Because it kept forgetting their passwords and never stopped talking about its "big data" dreams!<|end|>


tensor([[200021,  60751,    668,    261,  41751,   1078,    261,  19121,  25392,
          40602,      0, 200020, 200019,  62915,     11,  60623,    261,   4207,
         110641,  41751,   1078,    261,  19121,  25392,  40602,   1402,  13903,
           2242,    290,  19121,  25392,  40602,   2338,    869,    483,    290,
           7595,   1715,  30105,    480,  13185, 108180,   1043,  64653,    326,
           3779,  18145,  11695,   1078,   1617,    392,  25934,   1238,      1,
          23749,      0, 200020]], device='cuda:0')

Try other models as well:

* LLAMA = "meta-llama/Llama-3.2-1B-Instruct"
* PHI = "microsoft/Phi-4-mini-instruct"
* GEMMA = "google/gemma-3-270m-it"
* QWEN = "Qwen/Qwen3-4B-Instruct-2507"
* DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"